# PROMPT ENGINEERING

Understanding


*   Chain of Thought
*   Self Consistency
*   Tree of Thought
*   Group of Thought






In [ ]:
!pip -q install transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def ask_llm(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

    answer = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    return answer.strip()

In [ ]:
problem = """
If the train moves 60 kilometer per hour

for 2 hours,

How far will it go?

"""

print(problem)


If the train moves 60 kilometer per hour

for 2 hours,

How far will it go?




**1 Baseline Prompt**

In [ ]:
print("="*60)
print("BASELINE PROMPT")
print("="*60)

prompt = problem

print(ask_llm(prompt))

BASELINE PROMPT
To determine how far the train will travel in 2 hours at a speed of 60 kilometers per hour, you can use the formula for distance:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Given:
- Speed = 60 kilometers per hour
- Time = 2 hours

Substitute the given values into the formula:

\[ \text{Distance} = 60 \, \text{kilometers/hour} \times 2 \, \text{hours} \]
\[ \text{Distance} = 120 \, \text{kilometers} \]

So, the train will travel 120 kilometers in 2 hours.


**No Explaination**

In [ ]:
print("="*60)
print("BASELINE PROMPT")
print("="*60)

prompt = f"""
Answer only.

{problem}
"""

print(ask_llm(prompt))

BASELINE PROMPT
The distance traveled is calculated as follows:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Given:
- Speed = 60 km/hour
- Time = 2 hours

So,
\[ \text{Distance} = 60 \, \text{km/hour} \times 2 \, \text{hours} = 120 \, \text{km} \]

Therefore, the train will travel a total distance of **120 kilometers**.


**2 Zero-Shot Prompting**

In [ ]:
print("="*60)
print("ZERO SHOT PROMPTING")
print("="*60)

prompt = f"""
You are an expert mathematician.

Solve the following question.

Question

{problem}

Answer
"""

print(ask_llm(prompt))

ZERO SHOT PROMPTING
To solve this problem, we need to use the formula for distance traveled:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Given:
- Speed of the train = 60 kilometers per hour
- Time taken = 2 hours

We can substitute these values into the formula:

\[ \text{Distance} = 60 \, \text{kilometers/hour} \times 2 \, \text{hours} \]

Let's calculate the distance:

\[ \text{Distance} = 120 \, \text{kilometers} \]

So, the train will travel 120 kilometers in 2 hours.


**3 Better Zero Shot**

In [ ]:
print("="*60)
print("ZERO SHOT (STRICT)")
print("="*60)

prompt = f"""
You are an expert math solver.

Read carefully.

Return ONLY the final answer.

Question

{problem}

Final Answer
"""

print(ask_llm(prompt))

ZERO SHOT (STRICT)
120 kilometers


**4 Few Shot Prompting**

In [ ]:
print("="*60)
print("FEW SHOT")
print("="*60)

prompt = """
Question:
Ali has 2 apples and 3 oranges.

How many fruits he have in basket?

Answer:
5

Question:
Sara has 10 chocolates.

She eats 4.

Answer:
6

Question:

If the train moves 60 kilometer per hour

for 2 hours,

How far will it go?
Answer:
"""

print(ask_llm(prompt))

FEW SHOT
The answer is 120 kilometers.


**5 Chain of Thought**

In [ ]:
print("="*60)
print("CHAIN OF THOUGHT")
print("="*60)

prompt = f"""
Solve the problem.

Think step by step.

Question

{problem}

Answer
"""

print(ask_llm(prompt))

CHAIN OF THOUGHT
To solve this problem, we need to calculate the distance traveled by the train based on its speed and time. The formula for distance is:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Given:
- Speed of the train = 60 kilometers per hour
- Time taken = 2 hours

Step-by-step solution:

1. Identify the given values:
   - Speed (s) = 60 km/hour
   - Time (t) = 2 hours

2. Use the formula for distance:
   \[ \text{Distance} = s \times t \]
   where \( s \) is the speed in kilometers per hour and \( t \) is the time in hours.

3. Substitute the given values into the formula:
   \[ \text{Distance} = 60 \, \text{km/hour} \times 2 \, \text{hours} \]

4. Perform the multiplication:
   \[ \text{Distance} = 120 \, \text{km} \]

Therefore, the train will travel a distance of **120 kilometers**.


**6 Zero-Shot CoT**

This is the famous paper prompt.

Only one sentence changes.

In [ ]:
print("="*60)
print("ZERO SHOT CoT")
print("="*60)

prompt = f"""
{problem}

Let's think step by step.
"""

print(ask_llm(prompt))

ZERO SHOT CoT
To determine how far the train will travel given its speed and time, we can use the formula for distance:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Here, the speed of the train is 60 kilometers per hour (km/h), and the time it travels is 2 hours.

Step-by-step calculation:

1. Identify the speed: \( \text{Speed} = 60 \) km/h
2. Identify the time: \( \text{Time} = 2 \) hours
3. Multiply the speed by the time to find the distance:

\[ \text{Distance} = 60 \, \text{km/h} \times 2 \, \text{hours} \]
\[ \text{Distance} = 120 \, \text{km} \]

Therefore, the train will travel a total distance of 120 kilometers.


**7 Self Consistency**

In [ ]:
def ask_sample(prompt):

    messages = [
        {
            "role":"user",
            "content":prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.8,
        top_p=0.9
    )

    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

In [ ]:
prompt = f"""
{problem}

Let's think step by step.

At the end, write your answer in exactly this format:

Final Answer: <answer>
"""

for i in range(5):

    print("="*50)
    print("Reasoning Path", i+1)
    print("="*50)

    print(ask_sample(prompt))

Reasoning Path 1
The final answer is: 120 kilometers.
To find the distance traveled by the train, we can use the formula:
Distance = Speed × Time
In this case, the speed of the train is 60 km/h and the time is 2 hours.
So, the final answer is: 120 kilometers.
Reasoning Path 2
To determine how far the train travels at an initial speed of 60 kilometers per hour for 2 hours, we need to use the formula for distance traveled:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Given:
- Speed = 60 km/hour
- Time = 2 hours

Step-by-step calculation:

1. Identify the given values:
   - Speed = 60 km/hour
   - Time = 2 hours

2. Use the formula:
   \[ \text{Distance} = \text{Speed} \times \text{Time} \]

3. Substitute the given values into the formula:
   \[ \text{Distance} = 60 \, \text{km/hour} \times 2 \, \text{hours} \]

4. Perform the multiplication:
   \[ \text{Distance} = 120 \, \text
Reasoning Path 3
To calculate the distance traveled by the train over 2 hours at a speed of 60 kil

In [ ]:
from collections import Counter

answers = []

for i in range(5):

    print("="*50)
    print("Reasoning Path", i+1)
    print("="*50)

    response = ask_sample(prompt)
    print(response)

    answers.append(response)

Reasoning Path 1
The train will have traveled 120 kilometers in total.
Explanation:
- The train is moving at a speed of 60 km/h for 2 hours.
- Distance = Speed × Time = 60 km/h × 2 h = 120 km.
Therefore, the final answer is: Final Answer: 120 kilometers.
Reasoning Path 2
To determine how far the train will travel, we need to multiply its speed by the time it spends traveling.

Given:
- Speed = 60 kilometers per hour
- Time = 2 hours

Formula for distance:
\[ \text{Distance} = \text{Speed} \times \text{Time} \]

Step-by-step calculation:
1. Multiply the speed (in kilometers per hour) by the time (in hours):
   \[ 60 \, \text{kilometers/hour} \times 2 \, \text{hours} = 120 \, \text{kilometers} \]

Therefore, if the train travels at a speed of 60 kilometers per hour for 2 hours, it will cover a distance of **120 kilometers**.
Reasoning Path 3
To determine how far the train will travel if it moves at a speed of 60 kilometers per hour for 2 hours, we can use the formula for distance travele

In [ ]:
import re
from collections import Counter

print("\n" + "="*60)
print("FINAL ANSWERS FROM EACH PATH")
print("="*60)

final_answers = []

for i, response in enumerate(answers, 1):

    # Look for "Final Answer: ..."
    match = re.search(
        r"Final\s*Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE
    )

    if match:
        answer = match.group(1).strip()

    else:
        # Fallback: extract the last number if the expected format is missing
        numbers = re.findall(r"\d+(?:\.\d+)?", response)

        if numbers:
            answer = numbers[-1]
        else:
            answer = "Unknown"

    final_answers.append(answer)

    print(f"Path {i}: {answer}")

# ----------------------------
# Majority Voting
# ----------------------------

vote = Counter(final_answers)

most_common_answer, votes = vote.most_common(1)[0]

print("\n" + "="*60)
print("MAJORITY VOTING")
print("="*60)

for ans, count in vote.items():
    print(f"{ans} : {count} vote(s)")

print("\n" + "-"*60)
print(f"Final Selected Answer : {most_common_answer}")
print(f"Majority Votes        : {votes}/{len(final_answers)}")
print("-"*60)


FINAL ANSWERS FROM EACH PATH
Path 1: 120 kilometers.
Path 2: 120
Path 3: 120
Path 4: 120 kilometers
Path 5: 120

MAJORITY VOTING
120 kilometers. : 1 vote(s)
120 : 3 vote(s)
120 kilometers : 1 vote(s)

------------------------------------------------------------
Final Selected Answer : 120
Majority Votes        : 3/5
------------------------------------------------------------


In [ ]:
import re
from collections import Counter

print("\n" + "="*60)
print("FINAL ANSWERS FROM EACH PATH")
print("="*60)

final_answers = []

for i, response in enumerate(answers, 1):

    # ----------------------------------------
    # First try to extract "Final Answer:"
    # ----------------------------------------
    match = re.search(
        r"Final\s*Answer\s*:\s*(.+)",
        response,
        flags=re.IGNORECASE
    )

    if match:
        raw_answer = match.group(1).strip()

        # Extract the FIRST number after "Final Answer:"
        num = re.search(r"\d+(?:\.\d+)?", raw_answer)

        if num:
            answer = num.group(0)
        else:
            answer = raw_answer

    else:
        # ----------------------------------------
        # Fallback:
        # No "Final Answer:" found.
        # Take the LAST number from the entire response.
        # ----------------------------------------
        numbers = re.findall(r"\d+(?:\.\d+)?", response)

        if numbers:
            answer = numbers[-1]
        else:
            answer = "Unknown"

    final_answers.append(answer)

    print(f"Path {i}: {answer}")

# ----------------------------------------
# Majority Voting
# ----------------------------------------

vote = Counter(final_answers)

most_common_answer, votes = vote.most_common(1)[0]

print("\n" + "="*60)
print("MAJORITY VOTING")
print("="*60)

for ans, count in vote.items():
    print(f"{ans} : {count} vote(s)")

print("\n" + "-"*60)
print(f"Final Selected Answer : {most_common_answer}")
print(f"Majority Votes        : {votes}/{len(final_answers)}")
print("-"*60)


FINAL ANSWERS FROM EACH PATH
Path 1: 120
Path 2: 120
Path 3: 120
Path 4: 120
Path 5: 120

MAJORITY VOTING
120 : 5 vote(s)

------------------------------------------------------------
Final Selected Answer : 120
Majority Votes        : 5/5
------------------------------------------------------------


**Tree of Thoughts (ToT)**

In [ ]:
def generate_thought(problem):

    prompt = f"""
You are solving a reasoning problem using the Tree of Thoughts (ToT) approach.

Your task is to generate ONE possible reasoning path.

Instructions:
- Think step by step.
- Explore one possible solution.
- Do not assume this is the only correct reasoning.
- This reasoning will later be compared with other candidate thoughts.
- End with:

Final Answer: <your answer>

Problem:
{problem}
"""

    return ask_llm(prompt)

In [ ]:
def generate_multiple_thoughts(problem, n=3):

    thoughts = []

    for i in range(n):

        prompt = f"""
You are solving a reasoning problem using the Tree of Thoughts (ToT) approach.

Generate ONE independent reasoning path.

Instructions:
- Think step by step.
- Explore a possible solution.
- This is only ONE candidate thought.
- Different reasoning paths will be generated and compared later.
- Focus on logical and consistent reasoning.
- End with:

Final Answer: <your answer>

Problem:
{problem}
"""

        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            do_sample=True,
            temperature=0.9,
            top_p=0.9,
            max_new_tokens=250
        )

        answer = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )

        thoughts.append(answer)

    return thoughts

In [ ]:
thoughts = generate_multiple_thoughts(problem, n=3)

print("=" * 80)
print("TREE OF THOUGHTS : GENERATED CANDIDATE REASONING PATHS")
print("=" * 80)

for i, thought in enumerate(thoughts, start=1):

    print(f"\n{'=' * 80}")
    print(f"THOUGHT PATH {i}")
    print(f"{'=' * 80}")

    print(thought)

TREE OF THOUGHTS : GENERATED CANDIDATE REASONING PATHS

THOUGHT PATH 1
1. **Step-by-step analysis**:
   - The train's speed is given as 60 kilometers per hour for 2 hours.
   - Speed = Distance / Time.
   - Distance = Speed × Time.

2. **Calculating distance**:
   - Using the formula \( \text{Distance} = \text{Speed} \times \text{Time} \):
     \[
     \text{Distance} = 60 \, \text{kilometers/hour} \times 2 \, \text{hours}
     \]

3. **Performing the calculation**:
   - First, multiply the speed by the time:
     \[
     60 \times 2 = 120 \, \text{kilometers}
     \]
   - Since this is the distance to be traveled without considering additional distances, we do not need to divide by any other units unless instructed otherwise.

4. **Conclusion**:
   - Therefore, the train will travel a total of 120 kilometers.

The final answer: The train will travel a distance of **120 kilometers**.

THOUGHT PATH 2
1. The question asks for the distance traveled by a train that covers 60 kilometers in 

**Graph of Thoughts (GoT)**

In [ ]:
def initial_thoughts(problem):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

Your task is to generate EXACTLY TWO different high-level solution ideas.

IMPORTANT RULES:
- Do NOT solve the problem.
- Do NOT calculate the final answer.
- Do NOT provide step-by-step reasoning.
- Each idea should describe only a possible approach.
- The two ideas should use different reasoning strategies.

Problem:
{problem}

Return ONLY in the following format:

Idea 1:
<one or two sentences describing the first approach>

Idea 2:
<one or two sentences describing the second approach>
"""

    return ask_llm(prompt)

In [ ]:
def expand_idea(problem, idea):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

Expand the given solution idea into a detailed reasoning path.

IMPORTANT RULES:
- Expand ONLY the given idea.
- Do NOT introduce a completely new approach.
- Keep the reasoning consistent with the original idea.
- Explain the reasoning step by step.
- Do NOT provide the final answer.
- Do NOT conclude the problem.

Problem:
{problem}

Solution Idea:
{idea}

Return ONLY in the following format:

Expanded Reasoning:
<step-by-step reasoning based only on the given idea>
"""

    return ask_llm(prompt)

In [ ]:
def merge_ideas(problem, idea1, idea2):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

Your task is to combine the two reasoning paths into ONE stronger reasoning path.

IMPORTANT RULES:
- Do NOT choose one idea and ignore the other.
- Identify the strengths of BOTH reasoning paths.
- Merge their useful information into one coherent reasoning.
- Remove any redundant or contradictory statements.
- Preserve the logical flow.
- Do NOT generate a completely new approach.
- Do NOT provide the final answer yet.

Problem:
{problem}

Expanded Reasoning 1:
{idea1}

Expanded Reasoning 2:
{idea2}

Return ONLY in the following format:

Merged Reasoning:
<one improved reasoning path that combines both ideas>
"""

    return ask_llm(prompt)

In [ ]:
def final_answer(problem, merged):

    prompt = f"""
You are working in a Group of Thoughts (GoT) reasoning framework.

The reasoning from multiple solution ideas has already been generated,
expanded, and merged into one improved reasoning path.

Your task is to:

1. Use ONLY the merged reasoning below.
2. Solve the problem using that reasoning.
3. Do NOT generate a new approach.
4. Do NOT ignore the merged reasoning.
5. Clearly explain the final reasoning.
6. Provide the final answer.

Problem:
{problem}

Merged Reasoning:
{merged}

Return ONLY in the following format:

Final Reasoning:
<brief reasoning based on the merged reasoning>

Final Answer:
<answer only>
"""

    return ask_llm(prompt)

In [ ]:
ideas = initial_thoughts(problem)

print("\n" + "=" * 80)
print("GROUP OF THOUGHTS (GoT) : INITIAL IDEA GENERATION")
print("=" * 80)

print("\nThe model generated multiple high-level solution ideas.")
print("These are NOT complete solutions.")
print("They represent different approaches that will be expanded and merged later.\n")

print("=" * 80)
print("GENERATED INITIAL IDEAS")
print("=" * 80)

print(ideas)

print("\n" + "=" * 80)
print("NEXT STEP")
print("=" * 80)

print("Each idea will now be expanded into a detailed reasoning path.")


GROUP OF THOUGHTS (GoT) : INITIAL IDEA GENERATION

The model generated multiple high-level solution ideas.
These are NOT complete solutions.
They represent different approaches that will be expanded and merged later.

GENERATED INITIAL IDEAS
Idea 1: 
Calculate distance using formula: Distance = Speed × Time

Idea 2: 
Use proportionality between speed and time to find distance directly from the given information. 

Both approaches utilize basic mathematical principles but differ in their specific application. Idea 1 involves applying a direct calculation based on the provided data, while Idea 2 leverages a proportional relationship to derive the unknown quantity.

NEXT STEP
Each idea will now be expanded into a detailed reasoning path.
